In [1]:
import math
import platform
import sys
from dataclasses import dataclass
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import torch

# Reproducible numerical examples.
torch.manual_seed(7)
np.random.seed(7)
torch.set_default_dtype(torch.float64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
required_torch_func = ("jvp", "vjp", "grad", "vmap", "hessian")
missing_torch_func = [name for name in required_torch_func if not hasattr(torch.func, name)]

print("Python version:", sys.version.split()[0])
print("Platform:", platform.platform())
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Selected device:", device)
print("Default dtype:", torch.get_default_dtype())
print("Required torch.func APIs available:", not missing_torch_func)

if missing_torch_func:
    raise RuntimeError(
        "This notebook requires torch.func APIs that are missing from the active Colab runtime: "
        + ", ".join(missing_torch_func)
        + ". Select a current Colab runtime version and restart the session."
    )

Python version: 3.13.15
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
PyTorch version: 2.11.0+cpu
NumPy version: 2.1.3
Selected device: cpu
Default dtype: torch.float64
Required torch.func APIs available: True


In [2]:
def toy_renderer(theta):
    x, y = theta
    return torch.stack([torch.sin(x) + 0.25*y, x*y, torch.exp(-0.5*x) + y])
theta_true = torch.tensor([0.7, -0.25])
noise = torch.tensor([0.015, -0.010, 0.020])
I_clean = toy_renderer(theta_true)
I_obs = I_clean + noise
residual = I_obs - I_clean

print("clean prediction: ", I_clean.numpy())
print("observed evidence: ", I_obs.numpy())
print("discrepancy: ", residual.numpy())

clean prediction:  [ 0.58171769 -0.175       0.45468809]
observed evidence:  [ 0.59671769 -0.185       0.47468809]
discrepancy:  [ 0.015 -0.01   0.02 ]


In [ ]:
sigma = 0.05
quadratic_term = 0.5 * torch.sum((I_obs - I_clean)**2 / sigma**2)
gaussian_nll = quadratic_term + I_obs.numel() * math.log(sigma * math.sqrt(2*math.pi))

print("scaled quadratic term: ", quadratic_term.item())
print("gaussian negative log-likelihood: ", gaussian_nll.item())
print("constant offset: ", (gaussian_nll-quadratic_term).item())

scaled quadratic term:  0.14500000000000024
gaussian negative log-likelihood:  -6.085381221047954
constant offset:  -6.2303812210479546
